# Lecture 4 Colab: Credit Default Classification

This notebook demonstrates a simple credit default prediction workflow using synthetic tabular data. It is teaching material, not a production credit model.


## Learning objectives

- Generate a synthetic credit default dataset.
- Train logistic regression and random forest classifiers.
- Evaluate models using accuracy, precision, recall, F1-score, ROC-AUC, and confusion matrix.
- Understand why a probability score is not the same as a business decision.
- Explore how changing the classification threshold changes false positives and false negatives.


## Connection to Lecture 4

Supports supervised learning, classification vs regression, credit default prediction, logistic regression, random forest, train/test split, evaluation metrics, and ROC-AUC.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, precision_recall_curve, auc
np.random.seed(42)


## Dataset explanation

The dataset is synthetic. The feature values are not real customer records and should only be used for teaching. The positive class is `default = 1`.


In [ ]:
feature_names = ["income_score", "debt_to_income", "credit_history_length", "past_delinquency_count", "loan_amount_score", "credit_utilization", "repayment_stability", "recent_credit_inquiries"]
X, y = make_classification(n_samples=3000, n_features=8, n_informative=5, n_redundant=1, weights=[0.85, 0.15], class_sep=1.0, random_state=42)
df = pd.DataFrame(X, columns=feature_names)
df["default"] = y
df.head()


## Exploratory data analysis

Inspect the dataset and class balance before modelling.


In [ ]:
print("Shape:", df.shape)
print(df["default"].value_counts(normalize=True).rename("proportion"))
df["default"].value_counts().sort_index().plot(kind="bar", color=["steelblue", "indianred"])
plt.xticks([0, 1], ["No default", "Default"], rotation=0)
plt.ylabel("Count")
plt.title("Default vs non-default count")
plt.show()


## Train/test split

We use stratification so the default rate is similar in training and test sets.


In [ ]:
X = df[feature_names]
y = df["default"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Training default rate:", round(y_train.mean(), 3))
print("Test default rate:", round(y_test.mean(), 3))


## Train two classifiers

Logistic regression is an interpretable baseline. Random forest can capture non-linear patterns.


In [ ]:
log_reg = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
log_reg.fit(X_train, y_train)
rf.fit(X_train, y_train)


## Evaluation helper

A classifier outputs a score or probability. A threshold converts the score into a business decision.


In [ ]:
def evaluate_model(model_name, model, X_test, y_test, threshold=0.5):
    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    print(f"{model_name} at threshold {threshold}")
    print("Accuracy :", round(accuracy_score(y_test, predictions), 3))
    print("Precision:", round(precision_score(y_test, predictions, zero_division=0), 3))
    print("Recall   :", round(recall_score(y_test, predictions, zero_division=0), 3))
    print("F1-score :", round(f1_score(y_test, predictions, zero_division=0), 3))
    print("ROC-AUC  :", round(roc_auc_score(y_test, probabilities), 3))
    cm = confusion_matrix(y_test, predictions)
    ConfusionMatrixDisplay(cm, display_labels=["No default", "Default"]).plot(cmap="Blues", values_format="d")
    plt.title(f"Confusion matrix: {model_name}")
    plt.show()
    return probabilities, predictions


## Evaluate models at threshold 0.5

Observe how each metric answers a different question.


In [ ]:
log_probs, log_preds = evaluate_model("Logistic Regression", log_reg, X_test, y_test, threshold=0.5)
rf_probs, rf_preds = evaluate_model("Random Forest", rf, X_test, y_test, threshold=0.5)


## ROC curves

ROC-AUC measures ranking quality across thresholds. It does not choose the final business threshold.


In [ ]:
plt.figure(figsize=(7, 5))
for name, probs in [("Logistic Regression", log_probs), ("Random Forest", rf_probs)]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=f"{name} AUC={roc_auc_score(y_test, probs):.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random baseline")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Precision-recall curves

Precision-recall curves are useful when the positive class is relatively rare.


In [ ]:
plt.figure(figsize=(7, 5))
for name, probs in [("Logistic Regression", log_probs), ("Random Forest", rf_probs)]:
    precision, recall, _ = precision_recall_curve(y_test, probs)
    plt.plot(recall, precision, label=f"{name} PR-AUC={auc(recall, precision):.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-recall curves")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Threshold tuning

Lower thresholds usually increase recall but may increase false positives. Higher thresholds usually increase precision but may miss more actual defaults.


In [ ]:
rows = []
for threshold in [0.2, 0.3, 0.5, 0.7]:
    pred = (rf_probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    rows.append({"threshold": threshold, "precision": precision_score(y_test, pred, zero_division=0), "recall": recall_score(y_test, pred, zero_division=0), "f1": f1_score(y_test, pred, zero_division=0), "false_positives": fp, "false_negatives": fn})
pd.DataFrame(rows).round(3)


## Business interpretation

| Error type | Meaning | Possible business cost |
|---|---|---|
| False positive | Model predicts default, borrower would not default | Reject or overprice a good borrower, lost revenue, unfair experience |
| False negative | Model predicts no default, borrower actually defaults | Credit loss, collection cost, risk exposure |


## What to observe

- Probability scores are not final decisions.
- Thresholds change false positives and false negatives.
- ROC-AUC helps evaluate ranking quality, not the business threshold.


## Common pitfalls

- Reporting accuracy only.
- Forgetting that default is the positive class.
- Treating 0.5 as automatically correct.
- Treating synthetic results as real credit evidence.


## Try it yourself

- Change `class_sep` in the synthetic dataset.
- Change random forest `max_depth`.
- Try threshold 0.4 or 0.6.
- Decide whether the bank should prefer higher recall or higher precision.


## Reflection questions

- Why is accuracy not enough?
- Which error is more costly for a conservative lender?
- Why does ROC-AUC not directly choose the final business threshold?
- Which model would you choose and why?


## Final summary

Binary classifiers output scores or probabilities. Metrics describe different types of model quality. Threshold choice connects model output to business decision.
